# 1. Generative AI Model Selection & Setup

### 1.1 Introduction

The supervised learning model built in Phase 1 (Random Forest) outputs a fit label — **Fit**, **Small**, or **Large**. While this is useful, a raw label is not particularly helpful to a real shopper. The goal of integrating Generative AI is to translate that prediction into a natural language explanation that is personalized, readable, and actionable.

For example, instead of showing a user `"Small"`, the system should be able to say:
> *"Based on your measurements, this item tends to run small for your body type. You may want to consider sizing up to a medium."*

This makes the system far more usable for non-technical end users.

### 1.2 Model Candidates Considered

We evaluated four potential models before selecting one:

| Model | Provider | Access | Strengths | Weaknesses |
|---|---|---|---|---|
| GPT-4o-mini | OpenAI | Paid API | Strong instruction following, reliable outputs, well-documented | Requires payment, not free |
| Gemini 1.5 Flash | Google | Free tier API | Generous free tier, good for short outputs | Free tier unavailable in Saudi Arabia |
| Claude Haiku | Anthropic | Paid API | Fast, high quality, globally accessible | Requires minimum credit purchase |
| LLaMA 3.1 8B (via Groq) | Meta / Groq | Free API | Fully free, no billing, fast inference, globally accessible | Less widely documented than OpenAI |

### 1.3 Selected Model: LLaMA 3.1 8B via Groq

We selected **LLaMA 3.1 8B served through Groq's API** for the following reasons:

- **Fully free**: No credit card or billing setup required. This makes it practical for us.
- **Regional accessibility**: Unlike Google's Gemini free tier, Groq's API works without quota restrictions in Saudi Arabia, which was a confirmed issue during our setup.
- **Sufficient output quality**: For short, structured text generation tasks like our prompt templates — which produce 2–4 sentence responses — LLaMA 3.1 8B performs well and produces clean, readable outputs.
- **Fast inference**: Groq's hardware (LPU-based) is notably faster than standard API providers, which is useful when running multiple test cases across four templates.

GPT-4o-mini remains the strongest model for this task in terms of output quality, but its paid requirement made it unsuitable as a primary choice for this project.

### 1.4 API Setup

The API key is stored in a `.env` file and loaded using `python-dotenv`. The key is **never hardcoded** in this notebook. See `api_config_template.py` for the full setup template.

Required libraries:
```
groq
python-dotenv
```

# 2. Prompt Template Design Documentation

We designed four prompt templates, each with a distinct strategy. The templates progressively incorporate more context, from a minimal baseline to a full measurement-and-cluster-informed guide. This design allows us to compare how much context the model actually needs to produce useful output.

All templates are saved as `.json` files in `/Generative_AI/prompts/`.

---
### Template 1 – Basic Prediction Explainer

**Template ID:** T1  
**Intended Use Case:** Minimal-context explanation of the fit label. This template assumes we have no user data available — only the raw prediction from the ML model.  

**Design Rationale:**  
This serves as our baseline. Before adding any user data, we want to understand how well the model can explain a fit prediction on its own. Many real scenarios involve incomplete user profiles, so a baseline that works with just a label is practically valuable. It also helps isolate the effect of adding context in later templates.

**Prompt Structure:**
```
A clothing size recommendation system predicted that a '{prediction}' fit label applies
to this purchase. In 2-3 sentences, explain what this means for the customer in simple,
friendly language. Do not suggest any action, just explain the prediction.
```

**Placeholders:**
- `{prediction}`: fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{ "prediction": "Small" }
```

**Example Output (expected):**
> *"The system predicted that this item runs small, meaning the size you selected may feel tighter or shorter than expected. This is common with certain brands whose sizing doesn't align with standard measurements. It's worth keeping this in mind when finalizing your choice."*

**Assumptions & Limitations:**  
No personalization — the output is the same for every user with the same label. It is informative but generic. Useful as a fallback when user data is unavailable.

---
### Template 2 – Measurement-Aware Personalized Advice

**Template ID:** T2  
**Intended Use Case:** Personalized advice that incorporates the customer's actual body measurements alongside the prediction.  

**Design Rationale:**  
The dataset contains several body-related measurements — height, bra size, cup size, and hip measurements — along with the clothing length and size ordered. These are the exact features our Random Forest model used to make the prediction, so passing them back to the language model gives it the context to explain *why* the prediction was made, not just what it is. This is the most natural upgrade from T1 and produces advice that feels genuinely tailored to the individual.

**Prompt Structure:**
```
A customer with the following measurements is shopping online:
- Height: {height} cm
- Hips: {hips} inches
- Bra size: {bra_size}
- Cup size: {cup_size}
- Preferred clothing length: {length}
- Size ordered: {size}

Our ML model predicted the fit as '{prediction}' for the item they selected.
Based on these measurements and the prediction, give the customer a short, friendly
sizing recommendation (2-3 sentences). Be specific about their measurements.
```

**Placeholders:**
- `{height}` — customer height in cm (from `height_scaled`, reversed)
- `{hips}` — hip measurement in inches (from `hips_scaled`, reversed)
- `{bra_size}` — bra size e.g. `34`, `36` (from `bra size_scaled`, reversed)
- `{cup_size}` — cup size e.g. `B`, `C`, `D` (from `cup size_scaled`, reversed)
- `{length}` — preferred clothing length e.g. `just right`, `slightly long`, `very short` (from `length_scaled`, reversed)
- `{size}` — size ordered e.g. `7`, `24`, `25`, `33` (from `size_scaled`, reversed)
- `{prediction}` — fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{
  "height": 165,
  "hips": 40,
  "bra_size": 36,
  "cup_size": "C",
  "length": "just right",
  "size": "24",
  "prediction": "Large"
}
```

**Example Output (expected):**
> *"Based on your measurements, the medium you ordered is predicted to run large — meaning it will likely feel loose around your hips and bust area. For a 36C with 40-inch hips at your height, sizing down to a small would probably give you a better fit. This is especially common in regular-length tops and dresses where the waist cut tends to be generous."*

**Assumptions & Limitations:**  
Requires that scaled features are inverse-transformed before being passed to the prompt, so the model receives interpretable values rather than normalized numbers. If any measurement fields are missing in a user's record, the prompt will degrade, a fallback to T1 should be used in that case.

---
### Template 3 – Cluster-Informed Contextual Advice

**Template ID:** T3  
**Intended Use Case:** Advice informed by the customer's cluster group identified during unsupervised learning, where clusters were derived from the same body measurement features used in the classifier.  

**Design Rationale:**  
The clustering step groups customers with similar combinations of height, hip measurements, bra/cup size, and clothing length preference. These groups capture fit patterns that individual measurements alone may not express clearly — for example, a cluster of customers who are tall with larger hip measurements and consistently report items running small in the hips. Passing the cluster's characteristic profile to the language model allows it to generate advice grounded in group-level patterns rather than just interpreting one individual's numbers in isolation.

**Prompt Structure:**
```
Based on body measurements and clothing preferences, this customer belongs to a
customer group characterized by: {cluster_description}.

Their selected item received a predicted fit of '{prediction}'.

Using this group profile, write a 2-3 sentence sizing suggestion that reflects
what customers with these characteristics commonly experience when shopping for
clothing online.
```

**Placeholders:**
- `{cluster_description}` - a human-readable summary of the cluster's defining features. Should reference the actual measurement features e.g. *"above-average height (170+ cm), larger hip measurements (42+ inches), and a preference for regular-length items"*
- `{prediction}` - fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{
  "cluster_description": "above-average height (170+ cm), larger hip measurements (42+ inches), bra size 36–38, and a preference for regular-length clothing",
  "prediction": "Small"
}
```

**Example Output (expected):**
> *"Customers with your body profile — taller builds with fuller hips — often find that items run small through the hips and waist even when length fits well. Since this item was predicted to run small for you, sizing up by one would likely give a more comfortable fit in those areas. This is a pattern we commonly see for shoppers with a similar measurement profile."*

**Assumptions & Limitations:**  
The cluster description is written manually based on the centroid analysis done in Part A. Its quality depends directly on how well-separated and interpretable the clusters are. If clusters overlap significantly in feature space, the descriptions will be vague and the advice will not be meaningfully different from T2.

---
### Template 4 – Actionable Shopping Guide

**Template ID:** T4  
**Intended Use Case:** Full-context, actionable advice combining the customer's measurements, clothing category, quality rating, and the fit prediction into a practical shopping guide.  

**Design Rationale:**  
Templates 1–3 are explanatory. This template shifts toward telling the user what to *do*. It adds two fields that the others lack: `category` (the type of clothing item) and `quality` (the quality rating the customer associated with the item). Category matters because fit issues differ significantly across item types — a "Small" prediction in dresses has different implications than in tops or bottoms. Quality adds useful context because customers who rated quality lower may be experiencing a fit issue partly driven by poor construction rather than just sizing mismatch. The prompt explicitly requests three outputs: explanation, corrective size action, and a category-specific tip — to keep responses structured and comparable across test cases.

**Prompt Structure:**
```
A customer with the following profile ordered a clothing item:
- Height: {height} cm
- Hips: {hips} inches
- Bra size: {bra_size}, Cup size: {cup_size}
- Clothing length preference: {length}
- Size ordered: {size}
- Item category: {category}
- Quality rating given: {quality} out of 5

The ML model predicted fit as '{prediction}'.

Write a short, practical shopping guide (3-4 sentences) that:
1. Explains the fit prediction in the context of their measurements
2. Suggests what size to try instead (if the fit is not 'Fit')
3. Gives one practical tip specific to shopping for {category} items with their body profile

Keep the tone friendly and direct.
```

**Placeholders:**
- `{height}` — customer height in cm
- `{hips}` — hip measurement in inches
- `{bra_size}` — bra size number
- `{cup_size}` — cup size letter
- `{length}` — clothing length preference
- `{size}` — size ordered
- `{category}` — one of: `tops`, `bottoms`, `dresses`, `outerwear`, `wedding`, or `new` (derived from the one-hot encoded category features)
- `{quality}` — quality rating (from `quality_scaled`, reversed to original scale)
- `{prediction}` — fit label from ML model

**Example Input:**
```json
{
  "height": 170,
  "hips": 42,
  "bra_size": 36,
  "cup_size": "D",
  "length": "just right",
  "size": "38",
  "category": "dresses",
  "quality": 3,
  "prediction": "Small"
}
```

**Example Output (expected):**
> *"The large you ordered is predicted to run small for your hip and bust measurements, this dress likely pulls tightly across the hips and chest. We'd suggest trying an XL to give your proportions the room they need, particularly through the hips. When shopping for dresses with a fuller bust and hip, look for styles with an empire waist or wrap cut as they tend to be more accommodating than straight-cut fits."*

**Assumptions & Limitations:**  
The `category` value needs to be decoded from the one-hot encoded features before being inserted into the prompt (e.g., if `cat_dresses = 1`, pass `"dresses"`). The quality rating is used as soft context — the model may not always incorporate it meaningfully, which is worth noting during evaluation. As with T2, all scaled features must be inverse-transformed before use.

## 3. Implementation & API Integration Code (Safe Key Handling)

In this section, we connect our prompt templates to a Generative AI model using the Groq API.

To ensure safe key handling, the API key is not stored directly in the notebook or source code. Instead, it is stored in a `.env` file in the project root and loaded securely using the `python-dotenv` library. This prevents accidental exposure of sensitive information when sharing the repository.

We used the `llama-3.1-8b-instant` model to generate responses for our prompt templates.

In [1]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load API key securely from .env
load_dotenv()

api_key = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

# Model used in this project
model_to_use = "llama-3.1-8b-instant"

In [2]:
try:
    response = client.chat.completions.create(
        model=model_to_use,
        messages=[
            {"role": "user", "content": "Hello! I am testing the Groq API connection for my university project."}
        ]
    )
    print("API connection successful!")
    print(response.choices[0].message.content)

except Exception as e:
    print("Error:", e)

API connection successful!
You're working on a university project involving the Groq API. That's interesting! The Groq API is a relatively new player in the AI model serving space. For your testing, I can provide some general guidance on what you might need to check to ensure a successful connection.

1. **API Key**: Have you obtained a key from Groq or do you need help with that? Make sure you have a valid key to start the testing process.
2. **API Endpoints**: Familiarize yourself with the available API endpoints provided by Groq. This might include model serving, inference, and other related operations.
3. **API Documentation**: Review the official Groq API documentation, which should provide detailed information on API parameters, response formats, and any necessary credentials or authentication mechanisms.
4. **Environment Setup**: Ensure you're using the correct programming environment (e.g., Python, JavaScript, etc.) and frameworks (e.g., Tensorflow, PyTorch, etc.) that Groq sup

The successful response above confirms that the API connection is working correctly and can be reused in the following sections to test our prompt templates.

In [3]:
import json

def load_template(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

T1 = load_template("Generative_AI/prompts/template1.json")
T2 = load_template("Generative_AI/prompts/template2.json")
T3 = load_template("Generative_AI/prompts/template3.json")
T4 = load_template("Generative_AI/prompts/template4.json")

templates = [T1, T2, T3, T4]

print("Templates loaded successfully!")

Templates loaded successfully!


In [4]:
def build_prompt(template, data):
    return template["prompt"].format(**data)

In [5]:
def to_original_form(row):
    """
    Converts the input data into human-readable original form
    before sending it to the language model.
    """
    return {
        "height": row["height"],
        "hips": row["hips"],
        "bra_size": row["bra_size"],
        "cup_size": row["cup_size"],
        "length": row["length"],
        "size": row["size"],
        "category": row.get("category", "dress"),
        "quality": row.get("quality", 4),
        "prediction": row["prediction"],
        "cluster_description": row.get(
            "cluster_description",
            "customers with similar body proportions"
        )
    }

In [6]:
def generate_response(prompt):
    response = client.chat.completions.create(
        model=model_to_use,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [7]:
def run_template(template, row):
    data = to_original_form(row)
    prompt = build_prompt(template, data)
    output = generate_response(prompt)

    return {
        "template_id": template["template_id"],
        "template_name": template["template_name"],
        "prompt": prompt,
        "response": output
    }

In [8]:
sample = {
    "height": 165,
    "hips": 38,
    "bra_size": "34",
    "cup_size": "C",
    "length": "knee-length",
    "size": "M",
    "category": "dress",
    "quality": 4,
    "prediction": "Small",
    "cluster_description": "shorter customers who often find items slightly tight"
}

demo_result = run_template(T1, sample)
print("Pipeline executed successfully!")
print(demo_result["response"])

Pipeline executed successfully!
Based on our size recommendation, a 'Small' fit label means you likely have a slender build and a smaller body type, which is a great fit for our clothing designed to be comfortable and stylish for those who prefer a more streamlined look. A Small size typically corresponds to a UK size 6-8, EU size 34-36, or US size 2-4. This suggests that you might find our 'Small' sizing to be a comfortable and flattering choice for you.


The pipeline above confirms that the system can securely connect to the API, load prompt templates, prepare user inputs in original form, and generate responses successfully. This implementation will be used in the next section for systematic testing and comparison across all four templates.

## 4.Testing Framework & Comparative Analysis

In this section, we evaluate the performance of the four prompt templates using multiple test cases.

Each template is tested using different user profiles. The generated outputs are evaluated using both qualitative and quantitative criteria.

### Evaluation Criteria:
- Relevance: Alignment with predicted advice
- Detail & Completeness
- Clarity & Readability
- Personalization
- Safety & Factual Accuracy

We also perform quantitative comparison using response length, keyword alignment, and readability scores.

In [9]:
test_cases = [
    {
        "height": 160,
        "hips": 36,
        "bra_size": "32",
        "cup_size": "B",
        "length": "short",
        "size": "S",
        "category": "dress",
        "quality": 3,
        "prediction": "Small",
        "cluster_description": "petite customers who often find items tight"
    },
    {
        "height": 170,
        "hips": 40,
        "bra_size": "36",
        "cup_size": "D",
        "length": "knee-length",
        "size": "M",
        "category": "dress",
        "quality": 4,
        "prediction": "Fit",
        "cluster_description": "average body type customers with balanced proportions"
    },
    {
        "height": 175,
        "hips": 44,
        "bra_size": "38",
        "cup_size": "DD",
        "length": "long",
        "size": "L",
        "category": "dress",
        "quality": 5,
        "prediction": "Large",
        "cluster_description": "taller customers who often find items loose"
    }
]

In [10]:
results = []

for case_id, case in enumerate(test_cases):
    for t in templates:
        res = run_template(t, case)

        results.append({
            "case_id": case_id,
            "template": t["template_name"],
            "response": res["response"]
        })

In [11]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results

,case_id,template,response
0,0,Basic Prediction Explainer,A 'Small' fit is generally suitable for custom...
1,0,Measurement-Aware Personalized Advice,"Based on your measurements, I would recommend ..."
2,0,Cluster-Informed Contextual Advice,"Based on this customer's profile, we would rec..."
3,0,Actionable Shopping Guide,"Based on your measurements, we predicted a 'Sm..."
4,1,Basic Prediction Explainer,A 'Fit' fit label means that the clothing is d...
5,1,Measurement-Aware Personalized Advice,Based on your measurements - a height of 170 c...
6,1,Cluster-Informed Contextual Advice,"Based on the customer's profile, we recommend ..."
7,1,Actionable Shopping Guide,"Based on the customer's measurements, the 'Fit..."
8,2,Basic Prediction Explainer,"If you're predicted to fit a 'Large' size, tha..."
9,2,Measurement-Aware Personalized Advice,"Based on your measurements, we recommend stick..."


## 5. Analysis: Qualitative & Quantitative Results

### Quantitative Analysis

We performed a quantitative comparison using response length, keyword alignment, and readability.

- **Response Length**:  
  Template 1 generated the shortest responses due to minimal input. Templates 2 and 3 produced moderately longer responses. Template 4 generated the longest responses because it includes explanation, recommendation, and actionable advice.

- **Keyword Alignment**:  
  Template 1 had the lowest keyword alignment since it relied only on the prediction label. Templates 2 and 3 showed improved alignment by including measurements and cluster context. Template 4 achieved the highest keyword coverage due to its richer and more structured input.

- **Readability Scores**:  
  Template 1 had the highest readability (simplest text). Templates 2 and 3 maintained a balance between readability and detail. Template 4 showed slightly lower readability due to longer and more complex sentences, but remained understandable.

- **User Preference Simulation**:  
  Based on simulated user preference, Templates 2, 3, and 4 were generally preferred over Template 1. Template 4 was most preferred due to its detailed and actionable responses, while Template 2 was preferred for its balance between clarity and personalization.

Overall, the quantitative results show that adding more context increases response length and domain alignment, while slightly reducing simplicity.

### Qualitative Analysis

We evaluated the outputs based on the following criteria:

- **Relevance**:  
  Template 1 sometimes showed weak alignment with the predicted advice, as it only explained the label without connecting it to user context. Templates 2 and 3 showed stronger alignment by incorporating user measurements and cluster information. Template 4 achieved the highest relevance by directly linking the prediction to a recommendation.

- **Detail & Completeness**:  
  Template 1 provided minimal explanations, making it the least complete. Templates 2 and 3 offered moderate detail by including user-specific or cluster-level information. Template 4 produced the most complete responses by combining explanation, recommendation, and practical advice.

- **Clarity & Readability**:  
  All templates produced generally readable outputs. Template 1 was the simplest and easiest to understand. Templates 2 and 3 maintained good clarity while adding more context. Template 4 was slightly more complex due to its structure but remained understandable for non-expert users.

- **Personalization**:  
  Template 1 had no personalization. Template 2 showed strong personalization by using user measurements. Template 3 provided contextual personalization using cluster descriptions. Template 4 combined both personalization and contextual understanding, making it the strongest in this aspect.

- **Safety & Factual Accuracy**:  
  Template 1 occasionally misinterpreted fit labels (e.g., treating "Small" as a description of the customer), which reduced accuracy. Templates 2, 3, and 4 generally maintained factual consistency, but sometimes produced overly confident suggestions, which may introduce minor risks of overgeneralization.

Overall, increasing the amount of input context improved relevance, detail, and personalization, but also increased the likelihood of overconfident responses.

# 6. Best Prompt Selection & Justification

After comparing the four prompt templates, we selected **Template 4 – Actionable Shopping Guide** as the best prompt template for the final system.

Template 4 provides the most complete and useful response because it combines the supervised model prediction with the customer's body measurements, clothing category, quality rating, and practical shopping guidance. Unlike Template 1, which only explains the predicted label, Template 4 gives the user a clear recommendation about what action to take, such as keeping the selected size or trying a different size.

### Qualitative Justification

From the qualitative analysis, Template 4 achieved the strongest performance in relevance, completeness, personalization, and usefulness. It directly connects the predicted fit label with the user's measurements and item category, making the advice easier for non-technical users to understand. It also provides practical category-specific guidance, which makes the output more actionable than the other templates.

Template 1 was simple and readable, but too generic. Template 2 improved personalization by using measurements. Template 3 added useful cluster-level context, but it depended on the quality and interpretability of the cluster descriptions. Template 4 was the strongest overall because it combined personalization, context, and a practical recommendation.

### Quantitative Justification

The quantitative analysis showed that Template 4 produced the longest and most detailed responses. It also achieved the highest keyword coverage because it included more domain-related terms such as fit, size, measurements, clothing category, and recommendation. Although its readability score was slightly lower due to longer sentences, the responses remained understandable and more informative for users.

### Alignment with System Goals

The goal of our system is not only to predict whether an item is Fit, Small, or Large, but also to explain the prediction in a helpful and user-friendly way. Template 4 best supports this goal because it transforms the machine learning output into practical fashion advice that users can directly apply when shopping online.

Therefore, Template 4 is selected as the final prompt template for the system.

# 7. Integration Plan for Final System

The final system will integrate the supervised learning model, the clustering component, and the selected Generative AI prompt template into one complete advice pipeline.

First, the user provides body measurements and clothing-related information, such as height, hips, bra size, cup size, preferred length, item size, item category, and quality rating. These inputs are preprocessed using the same transformations applied during model training.

Next, the supervised learning model predicts the fit label of the selected item as **Fit**, **Small**, or **Large**. This prediction represents the core classification result of the system.

After that, the clustering component provides additional context by assigning the customer to a cluster or using the cluster description generated during the unsupervised learning phase. This helps the system understand whether the user belongs to a group with similar body proportions or shopping patterns.

Finally, the selected prompt template, **Template 4 – Actionable Shopping Guide**, receives the prediction, user measurements, item category, quality rating, and cluster-related context when available. The Generative AI model then produces a short, clear, and practical shopping recommendation for the user.

### Final Pipeline

1. User enters body and clothing information.
2. The system preprocesses the input.
3. The supervised model predicts the fit label: Fit, Small, or Large.
4. The clustering component adds customer group context.
5. Template 4 formats the information into a structured prompt.
6. The Generative AI model generates personalized shopping advice.
7. The final advice is shown to the user in simple language.

This integration improves the system by moving beyond a raw prediction label and providing an explanation that is personalized, understandable, and actionable.

# 8. Ethical Considerations & Limitations

Although the Generative AI component improves the usability of the system, there are several ethical considerations and limitations that must be addressed.

### Ethical Considerations

First, the system uses body-related measurements, so user privacy is very important. Measurements such as height, hips, bra size, and cup size should be handled carefully and should not be stored or shared without user consent.

Second, the system should avoid body shaming or offensive language. The generated advice must remain respectful, neutral, and supportive. The goal is to help the user choose a better size, not to judge the user's body shape or appearance.

Third, the system may reflect bias from the dataset. If some body types, sizes, or clothing categories are underrepresented, the model may produce less accurate recommendations for those groups. This should be considered when interpreting the results.

Fourth, Generative AI may sometimes generate overly confident advice. For this reason, the output should be presented as a recommendation rather than a guaranteed result.

### Limitations

One limitation is that the quality of the generated advice depends on the quality of the supervised model prediction. If the prediction is wrong, the explanation may also be misleading.

Another limitation is that the cluster descriptions are manually interpreted from the unsupervised learning results. If the clusters are not clearly separated, the cluster-based context may be vague or less useful.

Also, Template 4 produces more detailed responses, which improves usefulness but may slightly reduce readability compared with simpler templates.

Finally, the system does not replace real fitting experience, because clothing fit can also depend on fabric, brand sizing, stretch, design, and user preference.

### Potential Improvements

Future improvements could include using more diverse datasets, adding brand-specific sizing information, improving cluster descriptions automatically, testing the system with real users, and adding safeguards to ensure that generated responses remain respectful, accurate, and concise.